# Colab job worker (runs allow-listed notebooks that Claude Code submits through Drive)

Run this notebook once per Colab session, on the GPU you want the jobs to use (an **L4** for the pilot v2 and rival-benchmark notebooks). It mounts Drive, keeps a clean copy of the repository for itself, and polls `Drive/Colab Notebooks/AdjointRWM_Production/jobs/inbox/` for small JSON job files. Design and safety model: `docs/plans/colab-handoff.md` §5 and `src/adjointrwm/colab_jobs.py`.

**What a job can do.** A job names one notebook from `notebooks/05-ops/allowlist.json` **as it is on `origin/main`**, a full 40-character commit, a time limit, and a few whitelisted overrides (for example `seeds`). The worker refuses a job whose notebook is not on the list, whose override is not allowed, whose commit is not in the repository, or whose hardware does not match, and runs the notebook from that commit in a fresh kernel. A job cannot carry code, environment variables or secrets, and the worker never reads or prints any.

**What you see and control.** Every job's log is printed below. Results (an executed copy of the notebook, `result.json`, and each new run's summary files) go to `jobs/results/<job_id>/`; run directories go to `AdjointRWM_Production/runs/` as usual. To stop: use Colab's *Interrupt execution*, create a file named `STOP` in `jobs/`, or let the worker idle out (6 hours by default; the session budget is 10 hours). `jobs/worker_status.json` shows whether a worker is alive.

**Drive access.** The first cell asks you to authorise the Drive mount, which only you can do. Nothing here gives the worker access to anything outside what the mount already shares with Colab.

In [ ]:
# ================================================================
# 0. Mount Drive, get the worker's own clean checkout of main, print the hardware
# ================================================================
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/ROMUSKING/para_001'
WORKER_DIR = Path('/content/adjointrwm_worker')      # the worker's own copy; jobs run in /content/para_001, never here
if not WORKER_DIR.exists():
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(WORKER_DIR)], check=True)
subprocess.run(['git', '-C', str(WORKER_DIR), 'fetch', '--quiet', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(WORKER_DIR), 'checkout', '--quiet', '--detach', 'origin/main'], check=True)
WORKER_COMMIT = subprocess.check_output(['git', '-C', str(WORKER_DIR), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'nbclient', 'nbformat', 'ipykernel'], check=True)
print('worker code at', WORKER_COMMIT)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'no GPU in this runtime')

In [ ]:
# ================================================================
# 1. Run the worker (blocks; its log streams below). Interrupt the cell to stop it.
# ================================================================
DRIVE_ROOT = '/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production'
MAX_IDLE_HOURS = 6       # exit after this long without a job
MAX_SESSION_HOURS = 10   # never start a job that would outlast this budget

command = [sys.executable, str(WORKER_DIR / 'scripts' / 'colab_worker.py'), '--drive-root', DRIVE_ROOT, '--repo-dir', '/content/para_001',
           '--repo-url', REPO_URL, '--max-idle-hours', str(MAX_IDLE_HOURS), '--max-session-hours', str(MAX_SESSION_HOURS)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=str(WORKER_DIR))
try:
    for line in process.stdout:
        print(line, end='')
finally:
    if process.poll() is None:
        process.terminate()
        process.wait(timeout=60)
print('worker exited with code', process.returncode)